# Predizione della Popolarità Fotografica tramite Metadati EXIF

## 1. Introduzione del problema
L'obiettivo di questo progetto è determinare se i parametri tecnici di scatto scelti da un fotografo influenzano il successo e la popolarità di un'immagine. Nello specifico, si vuole costruire un algoritmo in grado di prevedere se una fotografia diventerà "Popolare" basandosi unicamente sui suoi metadati EXIF, senza analizzarne il contenuto visivo.

Il problema è formulato come un task di **Classificazione Binaria**:
* **Variabili Indipendenti (Features):** Sensibilità (ISO), Apertura del diaframma (f-stop), Tempo di esposizione (sec), Lunghezza focale (mm).
* **Variabile Dipendente (Target):** `is_popular` (1 se la foto rientra nel top 25% per numero di download, 0 altrimenti).

## 2. Descrizione del Dataset
Per questo progetto è stato utilizzato l'**Unsplash Dataset (Lite version)**, un dataset pubblico e gratuito rilasciato dalla piattaforma Unsplash. Esso contiene i metadati di oltre 25.000 fotografie reali.

Il dataset è fornito in formato TSV (Tab-Separated Values). Per il nostro scopo utilizzeremo due tabelle principali:
* `photos.tsv000`: Contiene l'ID univoco della foto e i parametri tecnici di scatto (`exif_iso`, `exif_aperture_value`, `exif_exposure_time`, `exif_focal_length`).
* `stats.tsv000`: Contiene l'ID della foto e le metriche di engagement (`downloads`, `views`).

## 3. Lettura e preparazione dei dati
In questa fase (come visto nella *Lezione 1* del corso), importeremo le librerie necessarie, caricheremo i dati, effettueremo il merge delle tabelle necessarie e faremo un primo preprocessing per gestire i valori mancanti.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

ASSETS_FOLDER_PATH = "assets"
PHOTOS_FILENAME = "photos.csv000"

# imposta lo stile per i grafici
sns.set_theme(style="whitegrid")

# verifica che i file sono nella cartella corretta
print("File presenti nella cartella:", os.listdir('.'))

try:
    # carica il dataframe e ne legge la dimensione per check
    df_photos = pd.read_csv(f'{ASSETS_FOLDER_PATH}/{PHOTOS_FILENAME}', sep='\t', header=0)
    print(f"\nFile {PHOTOS_FILENAME} letto con successo")
    print(f"Dimensioni dataset originale: {df_photos.shape}")
    
except FileNotFoundError as e:
    print("\nErrore: assicurarsi che i file csv000 siano nella cartella")
    print(e)

File presenti nella cartella: ['.git', 'requirements.txt', 'project.ipynb', 'assets', '.venv', '.gitignore']

File photos.csv000 letto con successo
Dimensioni dataset originale: (25000, 31)


In [ ]:

# trova automaticamente i nomi esatti delle colonne EXIF e statistiche presenti
colonne_exif_presenti = [col for col in df_photos.columns if 'exif' in col]
print(f"\nColonne EXIF trovate nel file: {colonne_exif_presenti}")

# usa i nomi esatti trovati, più i download e views
colonne_interesse = ['photo_id'] + colonne_exif_presenti + ['stats_downloads', 'stats_views']

# filtro del dataframe
colonne_effettive = [col for col in colonne_interesse if col in df_photos.columns]
df = df_photos[colonne_effettive]

# gestione dei dati mancanti
print("\nValori mancanti PRIMA del preprocessing")
print(df[colonne_exif_presenti].isnull().sum())

for col in colonne_exif_presenti:
    # forzamento della colonna a numerico
    df[col] = pd.to_numeric(df[col], errors='coerce')
    # calcolo della mediana di quella specifica colonna
    mediana = df[col].median()
    # riempiamo i buchi (NaN) con la mediana
    df[col] = df[col].fillna(mediana)

print("\nValori mancanti DOPO il preprocessing")
print(df[colonne_exif_presenti].isnull().sum())

# creazione della variabile target
if 'stats_downloads' in df.columns:
    soglia_popolarita = df['stats_downloads'].quantile(0.75)
    print(f"\nSoglia per definire una foto 'Popolare' (Top 25%): {soglia_popolarita} downloads")
    df['is_popular'] = (df['stats_downloads'] >= soglia_popolarita).astype(int)
else:
    print("\nErrore: Colonna stats_downloads non trovata! Controlla i nomi delle colonne di df")

print(f"\nDimensioni dataset finali pronte per l'analisi: {df.shape}")
display(df.head())


Colonne EXIF trovate nel file: ['exif_camera_make', 'exif_camera_model', 'exif_iso', 'exif_aperture_value', 'exif_focal_length', 'exif_exposure_time']

--- Valori mancanti PRIMA del preprocessing ---
exif_camera_make       2861
exif_camera_model      2906
exif_iso               3268
exif_aperture_value    3663
exif_focal_length      3552
exif_exposure_time     3288
dtype: int64

--- Valori mancanti DOPO il preprocessing ---
exif_camera_make       25000
exif_camera_model          0
exif_iso                   0
exif_aperture_value        0
exif_focal_length          0
exif_exposure_time         0
dtype: int64

Soglia per definire una foto 'Popolare' (Top 25%): 12197.5 downloads

Dimensioni dataset finali pronte per l'analisi: (25000, 10)


,photo_id,exif_camera_make,exif_camera_model,exif_iso,exif_aperture_value,exif_focal_length,exif_exposure_time,stats_downloads,stats_views,is_popular
0,oSf8ePoG9NU,NaN,181.5,200.0,4.0,40.0,15.0,30072,4296224,1
1,DlsOa5moK4w,NaN,181.5,200.0,1.8,50.0,15.0,26411,8731049,1
2,XBGacbT3vXI,NaN,181.5,400.0,10.0,23.0,15.0,19250,2224540,1
3,FjikPptEbZg,NaN,181.5,200.0,22.0,18.0,15.0,421,45536,0
4,PXdBkNF8rlk,NaN,181.5,100.0,2.8,200.0,15.0,43442,7394191,1
